# 03. Application + кредитное бюро

## Цель

Кратко:
- создать признаки из `bureau.csv` и `bureau_balance.csv`;
- добавить только признаки бюро к `application_train`;
- обучить CatBoost и записать эксперимент `application_bureau`.



## 1. Импорты и пути



In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from src.config import (
    INTERIM_DATA_DIR as DATA_INTERIM_DIR,
    PROCESSED_DATA_DIR as DATA_PROCESSED_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import (
    get_catboost_device_config,
    get_catboost_gpu_count,
    print_catboost_device_info,
)


APPLICATION_PATH = find_data_file("application_train.csv")
BUREAU_PATH = find_data_file("bureau.csv")
BUREAU_BALANCE_PATH = find_data_file("bureau_balance.csv")
CLIENT_SPLIT_PATH = (
    DATA_PROCESSED_DIR / "client_split.csv"
)
BUREAU_FEATURES_PATH = DATA_INTERIM_DIR / "bureau_features.csv"
RANDOM_STATE = 42
CV_FOLDS = 3


### Устройство CatBoost


In [3]:
gpu_count = get_catboost_gpu_count()
catboost_device_config = get_catboost_device_config(
    gpu_count=gpu_count,
)
print_catboost_device_info(
    catboost_device_config,
    gpu_count=gpu_count,
)


Modeling environment: Google Colab
CatBoost GPU count: 1
CatBoost device: GPU
CatBoost GPU devices: 0


## 2. Загрузка application_train

Основная таблица содержит одну строку на клиента. Техническое значение
`365243` в `DAYS_EMPLOYED` заменяется пропуском.



In [4]:
application = pd.read_csv(APPLICATION_PATH)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Application:", application.shape)
display(application.head())



Application: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Загрузка данных кредитного бюро



In [5]:
bureau = pd.read_csv(BUREAU_PATH)
bureau_balance = pd.read_csv(BUREAU_BALANCE_PATH)

assert bureau["SK_ID_BUREAU"].is_unique

print("Bureau:", bureau.shape)
print("Bureau balance:", bureau_balance.shape)
display(bureau.head())



Bureau: (1716428, 17)
Bureau balance: (27299925, 3)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


## 4. Признаки на уровне кредита

Используются только кредиты, известные не позднее текущей заявки.
Отношение долга к сумме кредита рассчитывается с защитой от деления на
ноль.



In [6]:
bureau = bureau[
    bureau["DAYS_CREDIT"].le(0)
].copy()

bureau["IS_ACTIVE"] = (
    bureau["CREDIT_ACTIVE"].eq("Active")
).astype(int)

bureau["HAS_OVERDUE"] = (
    bureau["CREDIT_DAY_OVERDUE"].gt(0)
).astype(int)

bureau["DEBT_TO_CREDIT_RATIO"] = (
    bureau["AMT_CREDIT_SUM_DEBT"]
    / bureau["AMT_CREDIT_SUM"].replace(0, np.nan)
)



## 5. Агрегация bureau до клиента



In [7]:
bureau_features = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        BUREAU_CREDIT_COUNT=("SK_ID_BUREAU", "count"),
        BUREAU_ACTIVE_SHARE=("IS_ACTIVE", "mean"),
        BUREAU_OVERDUE_SHARE=("HAS_OVERDUE", "mean"),
        BUREAU_CREDIT_AMOUNT_TOTAL=("AMT_CREDIT_SUM", "sum"),
        BUREAU_CREDIT_AMOUNT_MEAN=("AMT_CREDIT_SUM", "mean"),
        BUREAU_DEBT_TOTAL=("AMT_CREDIT_SUM_DEBT", "sum"),
        BUREAU_DEBT_TO_CREDIT_MEAN=(
            "DEBT_TO_CREDIT_RATIO",
            "mean",
        ),
        BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"),
        BUREAU_MOST_RECENT_CREDIT_DAYS=("DAYS_CREDIT", "max"),
        BUREAU_OLDEST_CREDIT_DAYS=("DAYS_CREDIT", "min"),
    )
    .reset_index()
)

bureau_features[
    "BUREAU_MOST_RECENT_CREDIT_DAYS"
] *= -1

bureau_features[
    "BUREAU_OLDEST_CREDIT_DAYS"
] *= -1

assert bureau_features["SK_ID_CURR"].is_unique

print(bureau_features.shape)
display(bureau_features.head())



(305811, 11)


,SK_ID_CURR,BUREAU_CREDIT_COUNT,BUREAU_ACTIVE_SHARE,BUREAU_OVERDUE_SHARE,BUREAU_CREDIT_AMOUNT_TOTAL,BUREAU_CREDIT_AMOUNT_MEAN,BUREAU_DEBT_TOTAL,BUREAU_DEBT_TO_CREDIT_MEAN,BUREAU_MAX_DAYS_OVERDUE,BUREAU_MOST_RECENT_CREDIT_DAYS,BUREAU_OLDEST_CREDIT_DAYS
0,100001,7,0.428571,0.0,1453365.000,207623.571429,596686.5,0.282518,0,49,1572
1,100002,8,0.250000,0.0,865055.565,108131.945625,245781.0,0.136545,0,103,1437
2,100003,4,0.250000,0.0,1017400.500,254350.125000,0.0,0.000000,0,606,2586
3,100004,2,0.000000,0.0,189037.800,94518.900000,0.0,0.000000,0,408,1326
4,100005,3,0.666667,0.0,657126.000,219042.000000,568408.5,0.601256,0,62,373


## 6. Агрегация bureau_balance

Месячные статусы сначала агрегируются до кредита `SK_ID_BUREAU`, затем
кредиты — до клиента.



In [8]:
bureau_balance = bureau_balance[
    bureau_balance["MONTHS_BALANCE"].le(0)
].copy()

status_map = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "C": 0,
    "X": np.nan,
}

bureau_balance["STATUS_NUMERIC"] = (
    bureau_balance["STATUS"].map(status_map)
)

bureau_balance["HAS_OVERDUE_MONTH"] = (
    bureau_balance["STATUS"].isin(["1", "2", "3", "4", "5"])
).astype(int)

balance_by_credit = (
    bureau_balance
    .groupby("SK_ID_BUREAU")
    .agg(
        BB_MONTH_COUNT=("MONTHS_BALANCE", "count"),
        BB_MAX_STATUS=("STATUS_NUMERIC", "max"),
        BB_OVERDUE_MONTH_SHARE=("HAS_OVERDUE_MONTH", "mean"),
    )
    .reset_index()
)

assert balance_by_credit["SK_ID_BUREAU"].is_unique

balance_with_client = balance_by_credit.merge(
    bureau[["SK_ID_BUREAU", "SK_ID_CURR"]],
    on="SK_ID_BUREAU",
    how="inner",
    validate="one_to_one",
)

balance_features = (
    balance_with_client
    .groupby("SK_ID_CURR")
    .agg(
        BB_CREDIT_COUNT=("SK_ID_BUREAU", "count"),
        BB_MONTH_COUNT_TOTAL=("BB_MONTH_COUNT", "sum"),
        BB_MAX_STATUS=("BB_MAX_STATUS", "max"),
        BB_OVERDUE_MONTH_SHARE_MEAN=(
            "BB_OVERDUE_MONTH_SHARE",
            "mean",
        ),
    )
    .reset_index()
)

assert balance_features["SK_ID_CURR"].is_unique



## 7. Проверка и сохранение признаков



In [9]:
rows_before = len(bureau_features)

bureau_features = bureau_features.merge(
    balance_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(bureau_features) == rows_before
assert bureau_features["SK_ID_CURR"].is_unique
assert "TARGET" not in bureau_features.columns

bureau_features.to_csv(
    BUREAU_FEATURES_PATH,
    index=False,
)

print("Сохранено:", BUREAU_FEATURES_PATH)
print(bureau_features.shape)
display(bureau_features.head())



Сохранено: /content/drive/MyDrive/credit-scoring-system/data/interim/bureau_features.csv
(305811, 15)


,SK_ID_CURR,BUREAU_CREDIT_COUNT,BUREAU_ACTIVE_SHARE,BUREAU_OVERDUE_SHARE,BUREAU_CREDIT_AMOUNT_TOTAL,BUREAU_CREDIT_AMOUNT_MEAN,BUREAU_DEBT_TOTAL,BUREAU_DEBT_TO_CREDIT_MEAN,BUREAU_MAX_DAYS_OVERDUE,BUREAU_MOST_RECENT_CREDIT_DAYS,BUREAU_OLDEST_CREDIT_DAYS,BB_CREDIT_COUNT,BB_MONTH_COUNT_TOTAL,BB_MAX_STATUS,BB_OVERDUE_MONTH_SHARE_MEAN
0,100001,7,0.428571,0.0,1453365.000,207623.571429,596686.5,0.282518,0,49,1572,7.0,172.0,1.0,0.007519
1,100002,8,0.250000,0.0,865055.565,108131.945625,245781.0,0.136545,0,103,1437,8.0,110.0,1.0,0.255682
2,100003,4,0.250000,0.0,1017400.500,254350.125000,0.0,0.000000,0,606,2586,NaN,NaN,NaN,NaN
3,100004,2,0.000000,0.0,189037.800,94518.900000,0.0,0.000000,0,408,1326,NaN,NaN,NaN,NaN
4,100005,3,0.666667,0.0,657126.000,219042.000000,568408.5,0.601256,0,62,373,3.0,21.0,0.0,0.000000


## 8. Merge с application



In [10]:
modeling_data = application.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print("Application:", application.shape)
print("После добавления bureau:", modeling_data.shape)
print(
    "Добавлено признаков:",
    modeling_data.shape[1] - application.shape[1],
)



Application: (307511, 122)
После добавления bureau: (307511, 136)
Добавлено признаков: 14


## Чтение единого client split


In [11]:
if not CLIENT_SPLIT_PATH.exists():
    raise FileNotFoundError(
        "Сначала выполните notebooks/02_application_baseline.ipynb. "
        f"Ожидаемый файл: {CLIENT_SPLIT_PATH}"
    )

client_split = pd.read_csv(CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = modeling_data.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


split
train      246008
holdout     61503
Name: count, dtype: int64


## 10. Создание X и y

`TARGET`, идентификатор клиента и техническая колонка разделения не
передаются модели.



In [12]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

n_holdout = int(
    modeling_data["split"].eq("holdout").sum()
)

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Train:", X_train.shape)
print("Holdout clients (не используется):", n_holdout)


Train: (246008, 134)
Holdout clients (не используется): 61503


## 11. Подготовка данных для CatBoost

Категориальные пропуски заменяются строкой. Числовые `NaN` остаются без
изменений: CatBoost обрабатывает их самостоятельно.



In [13]:
categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

X_train_catboost = X_train.copy()
X_train_catboost[categorical_columns] = (
    X_train_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

print("Категориальных признаков:", len(categorical_columns))


Категориальных признаков: 16


## 12. Стратифицированная кросс-валидация



In [14]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 13. Pool для CatBoost



In [15]:
train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=categorical_columns,
)



## 14. Параметры CatBoost



In [16]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    **catboost_device_config,
}


## 15. Библиотечная CV CatBoost

OOF-предсказания не требуются, поэтому используется `catboost.cv()`
без ручного цикла по фолдам. Test в CV не участвует.



In [17]:
catboost_cv_results = catboost_cv(
    pool=train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=100,
    as_pandas=True,
    verbose=100,
)



/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU


Training on fold [0/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7046032	best: 0.7046032 (0)	total: 170ms	remaining: 2m 49s
100:	test: 0.7460460	best: 0.7460460 (100)	total: 11s	remaining: 1m 37s
200:	test: 0.7476233	best: 0.7476233 (199)	total: 19.3s	remaining: 1m 16s
300:	test: 0.7509931	best: 0.7510059 (297)	total: 29.2s	remaining: 1m 7s
400:	test: 0.7533238	best: 0.7533272 (397)	total: 37.6s	remaining: 56.1s
500:	test: 0.7550628	best: 0.7550838 (491)	total: 47.5s	remaining: 47.3s
600:	test: 0.7558185	best: 0.7558332 (589)	total: 57.4s	remaining: 38.1s
700:	test: 0.7568424	best: 0.7568517 (698)	total: 1m 6s	remaining: 28.3s
800:	test: 0.7570253	best: 0.7570269 (787)	total: 1m 15s	remaining: 18.8s
900:	test: 0.7574050	best: 0.7574061 (896)	total: 1m 25s	remaining: 9.36s
999:	test: 0.7578463	best: 0.7578465 (997)	total: 1m 33s	remaining: 0us
bestTest = 0.7578464746
bestIteration = 997
Training on fold [1/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7097206	best: 0.7097206 (0)	total: 226ms	remaining: 3m 45s
100:	test: 0.7498191	best: 0.7498191 (99)	total: 9.62s	remaining: 1m 25s
200:	test: 0.7519320	best: 0.7519320 (200)	total: 19.4s	remaining: 1m 17s
300:	test: 0.7542625	best: 0.7542625 (281)	total: 28.1s	remaining: 1m 5s
400:	test: 0.7555746	best: 0.7555758 (398)	total: 37.5s	remaining: 56s
500:	test: 0.7565889	best: 0.7565889 (499)	total: 47.2s	remaining: 47s
600:	test: 0.7581344	best: 0.7581344 (600)	total: 55.5s	remaining: 36.9s
700:	test: 0.7588951	best: 0.7588959 (687)	total: 1m 5s	remaining: 27.9s
800:	test: 0.7590553	best: 0.7590553 (792)	total: 1m 15s	remaining: 18.6s
900:	test: 0.7595929	best: 0.7596361 (856)	total: 1m 23s	remaining: 9.16s
999:	test: 0.7601779	best: 0.7601825 (998)	total: 1m 33s	remaining: 0us
bestTest = 0.7601824999
bestIteration = 998
Training on fold [2/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7048665	best: 0.7048665 (0)	total: 169ms	remaining: 2m 48s
100:	test: 0.7460661	best: 0.7460661 (100)	total: 10.6s	remaining: 1m 33s
200:	test: 0.7485960	best: 0.7485989 (198)	total: 19.6s	remaining: 1m 17s
300:	test: 0.7499416	best: 0.7499416 (296)	total: 28.8s	remaining: 1m 6s
400:	test: 0.7517732	best: 0.7518114 (399)	total: 38.6s	remaining: 57.7s
500:	test: 0.7535214	best: 0.7535214 (498)	total: 47.2s	remaining: 47s
600:	test: 0.7543679	best: 0.7543741 (596)	total: 56.8s	remaining: 37.7s
700:	test: 0.7549226	best: 0.7549226 (698)	total: 1m 6s	remaining: 28.4s
800:	test: 0.7556060	best: 0.7556093 (790)	total: 1m 14s	remaining: 18.6s
900:	test: 0.7563465	best: 0.7563497 (889)	total: 1m 24s	remaining: 9.33s
999:	test: 0.7570628	best: 0.7570638 (995)	total: 1m 35s	remaining: 0us
bestTest = 0.7570637763
bestIteration = 995


## 16. Лучшая итерация и CV-метрики



In [18]:
auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)

auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)

pr_auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)

pr_auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[
    auc_mean_column
].idxmax()

best_cv_row = catboost_cv_results.loc[
    best_cv_index
]

best_iteration = int(
    best_cv_row["iterations"]
) + 1

cv_roc_auc = float(
    best_cv_row[auc_mean_column]
)

cv_roc_auc_std = float(
    best_cv_row[auc_std_column]
)

cv_pr_auc = float(
    best_cv_row[pr_auc_mean_column]
)

cv_pr_auc_std = float(
    best_cv_row[pr_auc_std_column]
)

print(f"Лучшая итерация: {best_iteration}")
print(
    f"CV ROC-AUC: {cv_roc_auc:.4f} "
    f"± {cv_roc_auc_std:.4f}"
)
print(
    f"CV PR-AUC: {cv_pr_auc:.4f} "
    f"± {cv_pr_auc_std:.4f}"
)



Лучшая итерация: 1000
CV ROC-AUC: 0.7584 ± 0.0016
CV PR-AUC: nan ± nan


## 17. Итоговая модель CatBoost



In [19]:
catboost_model = CatBoostClassifier(
    iterations=best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


## 18. Обучение итоговой модели



In [20]:
catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)



Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 63.2ms	remaining: 1m 3s
100:	total: 4.57s	remaining: 40.7s
200:	total: 10.4s	remaining: 41.4s
300:	total: 14.8s	remaining: 34.4s
400:	total: 19.3s	remaining: 28.8s
500:	total: 25.5s	remaining: 25.4s
600:	total: 30s	remaining: 19.9s
700:	total: 34.5s	remaining: 14.7s
800:	total: 40.5s	remaining: 10.1s
900:	total: 44.9s	remaining: 4.94s
999:	total: 50.1s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, devices='0', eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

## Запись CV-результата


In [21]:
current_result = {
    "experiment": "application_bureau",
    "notebook": "03_bureau_features.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application + bureau",
    "source_tables": "application_train.csv, bureau.csv, bureau_balance.csv",
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": best_iteration,
    "cv_roc_auc": cv_roc_auc,
    "cv_roc_auc_std": cv_roc_auc_std,
    "cv_pr_auc": cv_pr_auc,
    "cv_pr_auc_std": cv_pr_auc_std,
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(current_result)
display(all_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN
2,application_bureau,03_bureau_features.ipynb,CatBoostClassifier,application + bureau,"application_train.csv, bureau.csv, bureau_bala...",GPU,246008,61503,134,3,1000.0,0.758362,0.001620,NaN,NaN,NaN,NaN


## Выводы



In [22]:
print("Эксперимент: application + bureau")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Лучшая итерация: {best_iteration}")
print(f"CV ROC-AUC: {cv_roc_auc:.4f}")
print(f"CV PR-AUC: {cv_pr_auc:.4f}")
print("Holdout не использовался: результат сравнивается только по CV.")


Эксперимент: application + bureau
Количество признаков: 134
Лучшая итерация: 1000
CV ROC-AUC: 0.7584
CV PR-AUC: nan
Holdout не использовался: результат сравнивается только по CV.
